<a href="https://colab.research.google.com/github/Abdulkadir03/Crop-Recommender-for-Sahel-Region/blob/main/notebooks/10_suitability_scores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10: EcoCrop suitability scores and a check of the limits

**Purpose:** Test whether the FAO EcoCrop limits (temperature, rainfall, soil pH) agree with where each crop is actually grown according to SPAM, turn them into a soft suitability score per crop, and check whether that score adds anything to the LightGBM models on field observations. The limits are priors, not hard filters.

**Inputs:** `grid.parquet` (01), `training_table.parquet` and `training_config.json` (07), `crop_requirements.parquet` (06), `oof_lgb_env.parquet` (09) and the ground-truth files from 05b and 05c (used in the re-validation step).

**Method:**
1. Aggregate the environment features to SPAM pixels (21,128 rows), as in notebooks 08 and 09.
2. Match each EcoCrop variable to a feature: mean temperature (`temp_mean`), annual rainfall (`chirps_annual_mean`), topsoil pH (`phh2o_top`). EcoCrop limits describe the growing season, so annual means are a proxy and this is stated as a limitation.
3. For each crop, compare the EcoCrop absolute and optimal ranges with the distribution of the feature in pixels where the crop is present (share of at least 5%) and where it is dominant: share of present pixels outside the absolute range, and quantiles of the observed values.
4. Build a trapezoidal suitability score per variable (0 outside the absolute range, 1 inside the optimal range, linear in between), with limits widened where the data show they are too narrow. Combine variables by the minimum (limiting factor) and, separately, by the mean.
5. Rice and other irrigation- or floodplain-dependent crops: add a water-access exception using `irrig_share`, since EcoCrop assumes rain-fed cultivation.
6. Re-validate: use the scores as extra features or as a soft prior on the LightGBM out-of-fold predictions, and compare with 09b (SPAM agreement, survey AUC with bootstrap intervals, CropHarvest top-5 hit rate). Any new model must keep the same spatial folds.

**Outputs** (Google Drive): `ecocrop_limit_check.csv`, `ecocrop_scores_pixel.parquet`,`ecocrop_flags.jason`, results tables.

**Notes:**
- EcoCrop values are expert-compiled and global; several limits are known to be too narrow for the Sahel (for example sorghum's absolute maximum rainfall of 700 mm).
- SPAM already contains GAEZ-based suitability logic, so agreement between EcoCrop and SPAM does not prove the limits are right. The field data (05b, 05c) remain the independent check.
- Sample sizes for rare crops are small; report counts of pixels next to every statistic.

**Results:**
Out-of-fold predictions for the random forest (09) and LightGBM (09) are compared with the EcoCrop scores in four checks. "Rain-fed" means the rainfall term applies to rice; "water access" means it is dropped for rice (the rule for when irrigation or floodplain access is available).

**1. Agreement with SPAM (all labelled pixels)**

| Method | SPAM rank corr | SPAM top-5 overlap | SPAM top-1 match |
|---|---|---|---|
| RF (09) | 0.644 | 0.789 | 0.591 |
| LightGBM (09) | 0.675 | 0.794 | 0.590 |
| EcoCrop min, rain-fed | 0.229 | 0.496 | 0.127 |
| EcoCrop min, water access | 0.237 | 0.481 | 0.132 |
| EcoCrop mean, rain-fed | 0.235 | 0.488 | 0.127 |
| LightGBM x EcoCrop min, rain-fed | 0.582 | 0.742 | 0.494 |
| LightGBM x EcoCrop min, water access | 0.610 | 0.737 | 0.507 |

**2. Survey presence AUC (Niger and Nigeria, crops with 10+ pixels per class)**

| Method | Mean AUC | Crop-countries with bootstrap CI above 0.5 | Crop-countries tested |
|---|---|---|---|
| EcoCrop mean, rain-fed | 0.590 | 9 | 18 |
| EcoCrop min, rain-fed | 0.590 | 8 | 17 |
| EcoCrop min, water access | 0.583 | 8 | 18 |
| LightGBM (09) | 0.658 | 12 | 18 |
| LightGBM x EcoCrop min, rain-fed | 0.637 | 9 | 17 |
| LightGBM x EcoCrop min, water access | 0.638 | 12 | 18 |
| RF (09) | 0.674 | 13 | 18 |

**3. CropHarvest top-5 hit rate (chance 0.31)**

| Method | All | Excluding five most common | Five most common | Sorghum | Rice |
|---|---|---|---|---|---|
| RF (09) | 0.765 | 0.218 | 0.944 | 1.000 | 0.057 |
| LightGBM (09) | 0.765 | 0.195 | 0.951 | 1.000 | 0.114 |
| EcoCrop min, rain-fed | 0.521 | 0.149 | 0.643 | 0.222 | 0.000 |
| EcoCrop min, water access | 0.615 | 0.529 | 0.643 | 0.222 | 0.943 |
| EcoCrop mean, rain-fed | 0.521 | 0.149 | 0.643 | 0.222 | 0.000 |
| LightGBM x EcoCrop min, rain-fed | 0.711 | 0.126 | 0.902 | 0.698 | 0.000 |
| LightGBM x EcoCrop min, water access | 0.725 | 0.207 | 0.895 | 0.698 | 0.200 |

The rice gain under water access (0.000 to 0.943) follows from the rule, which lifts rice in every dry pixel. It is not validation: the field data do not record irrigation status.

**4. Rice in rainfall-dry labelled pixels (below 1,000 mm)**

| Rule | Share of 3,801 rice-present pixels with score at least 0.5 |
|---|---|
| Rain-fed | 0.000 |
| Water access | 0.997 |

**Survey AUC by crop: largest differences from LightGBM (09)**

| Country | Crop | EcoCrop min, rain-fed | LightGBM (09) | LightGBM x EcoCrop min, rain-fed | RF (09) | Gap (EcoCrop minus LightGBM) |
|---|---|---|---|---|---|---|
| Niger | sorghum | 0.55 | 0.78 | 0.70 | 0.80 | -0.23 |
| Niger | tomato | 0.48 | 0.69 | 0.49 | 0.74 | -0.21 |
| Nigeria | groundnut | 0.48 | 0.67 | 0.66 | 0.64 | -0.19 |
| Niger | onion | 0.41 | 0.57 | 0.48 | 0.61 | -0.16 |
| Nigeria | cowpea | 0.48 | 0.62 | 0.61 | 0.69 | -0.14 |
| Niger | sesame | 0.56 | 0.68 | 0.66 | 0.68 | -0.12 |
| Nigeria | rice | 0.60 | 0.62 | 0.60 | 0.65 | -0.02 |

Full table: `results_ecocrop_survey_auc.csv` in `Drive`.

**Conclusion.** EcoCrop limits reduce agreement with SPAM and field AUC when used as a score or multiplier, so they are not used as a ranking term. They are used as feasibility flags with a stated reason (see Cell 5). The rice water-access rule is a warning, not a validated gain.



In [ ]:
# @title Cell 1: load the grid, the training table and the crop requirements
import os, json
import numpy as np, pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'

cfg = json.load(open(f'{P}/training_config.json'))
GROUPS, CROPS = cfg['groups'], cfg['crops']
ENV = GROUPS['climate'] + GROUPS['soil'] + GROUPS['terrain'] + GROUPS['salinity']
SHARE = [f'share_{c}' for c in CROPS]
PRESENT = [f'present_{c}' for c in CROPS]

grid = pd.read_parquet(f'{P}/grid.parquet')
tab = pd.read_parquet(f'{P}/training_table.parquet',
                      columns=['grid_id', 'px', 'py', 'fold', 'adm0', 'has_label', 'n_cells',
                               'weight', 'irrig_share'] + ENV + SHARE + PRESENT)
req = pd.read_parquet(f'{P}/crop_requirements.parquet').set_index('crop').loc[CROPS]
print('Grid:', grid.shape, '| training table:', tab.shape, '| requirements:', req.shape)
print('Same grid ids in grid and training table:', set(grid.grid_id) == set(tab.grid_id))

# Pixel table (one row per SPAM pixel), same construction as notebooks 08 and 09
lab = tab[tab.has_label]
g = lab.groupby(['px', 'py'])
pix = (g[ENV].mean().join(g[SHARE].first()).join(g[PRESENT].first())
       .join(g[['fold', 'irrig_share']].first())
       .join(g['adm0'].agg(lambda s: s.mode().iat[0])).reset_index())
print('Pixels:', len(pix), '| environment features:', len(ENV), '| crops:', len(CROPS))

# Which feature stands in for each EcoCrop variable
VARMAP = {'temperature': 'temp_mean', 'rainfall': 'chirps_annual_mean', 'ph': 'phh2o_top'}
LIMITS = {'temperature': ['tmin', 'topmn', 'topmx', 'tmax'],
          'rainfall':    ['rmin', 'ropmn', 'ropmx', 'rmax'],
          'ph':          ['phmin', 'phopmn', 'phopmx', 'phmax']}
print('\nFeature found for each variable:', {v: f in pix.columns for v, f in VARMAP.items()})
print('Observed range of those features over pixels (5th / 50th / 95th percentile):')
print(pix[list(VARMAP.values())].quantile([.05, .5, .95]).T.round(2))

# EcoCrop limits per crop, with the number of missing limits
cols = sum(LIMITS.values(), []) + ['gmin', 'gmax', 'ktmp']
pd.set_option('display.width', 250)
print('\nEcoCrop limits:')
print(req[cols].round(1).to_string())
print('\nMissing limit values per column:'); print(req[cols].isna().sum()[lambda s: s > 0])
print('Crops with all temperature, rainfall and pH limits present:',
      int(req[sum(LIMITS.values(), [])].notna().all(axis=1).sum()), 'of', len(req))

In [ ]:
# @title Cell 2: EcoCrop limits against where each crop is actually grown
# For each crop and variable: distribution of the matching feature in pixels where the crop is present
# (share >= 5%) and where it is dominant, compared with the EcoCrop absolute and optimal ranges.
MIN_N = 20                                    # smaller groups are reported as NaN, not summarised
dom_idx = pix[SHARE].to_numpy('float64').argmax(axis=1)

rows = []
for j, crop in enumerate(CROPS):
    masks = {'present': pix[f'present_{crop}'].to_numpy() > 0,
             'dominant': dom_idx == j}
    for var, feat in VARMAP.items():
        lo_abs, lo_opt, hi_opt, hi_abs = req.loc[crop, LIMITS[var]].astype(float)
        for grp, m in masks.items():
            x = pix.loc[m, feat].dropna().to_numpy('float64')
            d = {'crop': crop, 'variable': var, 'feature': feat, 'group': grp, 'n_pixels': len(x),
                 'abs_min': lo_abs, 'opt_min': lo_opt, 'opt_max': hi_opt, 'abs_max': hi_abs}
            if len(x) >= MIN_N:
                d.update({'p05': np.percentile(x, 5), 'p50': np.percentile(x, 50), 'p95': np.percentile(x, 95),
                          'pct_below_abs': (x < lo_abs).mean() * 100,
                          'pct_above_abs': (x > hi_abs).mean() * 100,
                          'pct_outside_abs': ((x < lo_abs) | (x > hi_abs)).mean() * 100,
                          'pct_outside_opt': ((x < lo_opt) | (x > hi_opt)).mean() * 100})
            rows.append(d)
chk = pd.DataFrame(rows)

# Reference: how much of the whole pixel set lies outside each crop's absolute range
ref = []
for crop in CROPS:
    for var, feat in VARMAP.items():
        lo_abs, _, _, hi_abs = req.loc[crop, LIMITS[var]].astype(float)
        x = pix[feat].dropna().to_numpy('float64')
        ref.append({'crop': crop, 'variable': var, 'pct_all_pixels_outside_abs': ((x < lo_abs) | (x > hi_abs)).mean() * 100})
chk = chk.merge(pd.DataFrame(ref), on=['crop', 'variable'])

pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200)
pres = chk[chk.group == 'present']
print('Pixels per crop: present / dominant')
print(pd.DataFrame({'present': pres[pres.variable == 'rainfall'].set_index('crop').n_pixels,
                    'dominant': chk[(chk.group == 'dominant') & (chk.variable == 'rainfall')].set_index('crop').n_pixels}).T.to_string())

print('\nShare of PRESENT pixels outside the EcoCrop ABSOLUTE range (%), by variable:')
print(pres.pivot(index='crop', columns='variable', values='pct_outside_abs').round(1).loc[CROPS].to_string())

print('\nShare of PRESENT pixels outside the EcoCrop OPTIMAL range (%), by variable:')
print(pres.pivot(index='crop', columns='variable', values='pct_outside_opt').round(1).loc[CROPS].to_string())

print('\nRainfall (mm): EcoCrop limits against observed quantiles in pixels where the crop is present')
rain = pres[pres.variable == 'rainfall'].set_index('crop').loc[CROPS]
print(rain[['n_pixels', 'abs_min', 'opt_min', 'opt_max', 'abs_max', 'p05', 'p50', 'p95',
            'pct_below_abs', 'pct_above_abs']].round(0).to_string())

print('\npH: EcoCrop limits against observed quantiles (present pixels)')
ph = pres[pres.variable == 'ph'].set_index('crop').loc[CROPS]
print(ph[['n_pixels', 'abs_min', 'opt_min', 'opt_max', 'abs_max', 'p05', 'p50', 'p95', 'pct_outside_abs']].round(2).to_string())

print('\nTemperature (annual mean, degC): limits against observed quantiles (present pixels)')
tm = pres[pres.variable == 'temperature'].set_index('crop').loc[CROPS]
print(tm[['n_pixels', 'abs_min', 'opt_min', 'opt_max', 'abs_max', 'p05', 'p50', 'p95', 'pct_outside_abs']].round(1).to_string())

# Crops where more than 10% of present pixels (and at least MIN_N pixels) fall outside the absolute range
flag = pres[(pres.pct_outside_abs > 10) & (pres.n_pixels >= MIN_N)][
    ['crop', 'variable', 'n_pixels', 'pct_outside_abs', 'pct_all_pixels_outside_abs']].round(1)
print('\nCrop-variable pairs with more than 10% of present pixels outside the absolute range:')
print(flag.sort_values('pct_outside_abs', ascending=False).to_string(index=False))

chk.to_csv(f'{P}/ecocrop_limit_check.csv', index=False)
drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 3: proposed widened limits, water-access check and field check of the limits
WIDE_Q, OPT_Q = (2, 98), (10, 90)     # percentiles of the present-pixel distribution (absolute, optimal)

# ---- Part A: widened limits (rainfall and pH only; limits are only ever widened, never narrowed) ----
prop = []
for crop in CROPS:
    m = pix[f'present_{crop}'].to_numpy() > 0
    for var in ['rainfall', 'ph']:
        x = pix.loc[m, VARMAP[var]].dropna().to_numpy('float64')
        a0, o0, o1, a1 = req.loc[crop, LIMITS[var]].astype(float)
        step = 10 if var == 'rainfall' else 0.1
        qa, qo = np.percentile(x, WIDE_Q), np.percentile(x, OPT_Q)
        fl = lambda v: round(float(np.floor(v / step) * step), 1)
        ce = lambda v: round(float(np.ceil(v / step) * step), 1)
        prop.append({'crop': crop, 'variable': var, 'n_present': len(x),
                     'abs_min': a0, 'abs_min_new': min(a0, fl(qa[0])),
                     'opt_min': o0, 'opt_min_new': min(o0, fl(qo[0])),
                     'opt_max': o1, 'opt_max_new': max(o1, ce(qo[1])),
                     'abs_max': a1, 'abs_max_new': max(a1, ce(qa[1]))})
prop = pd.DataFrame(prop)
changed = ((prop.abs_min_new < prop.abs_min) | (prop.abs_max_new > prop.abs_max) |
           (prop.opt_min_new < prop.opt_min) | (prop.opt_max_new > prop.opt_max))
WIDE = {(r.crop, r.variable): (r.abs_min_new, r.opt_min_new, r.opt_max_new, r.abs_max_new)
        for r in prop.itertuples()}
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 200)
print(f'Limits changed for {int(changed.sum())} of {len(prop)} crop-variable pairs. EcoCrop (abs_min, opt_min, opt_max, abs_max) vs widened (*_new):')
print(prop[changed].round(1).to_string(index=False))
prop.to_csv(f'{P}/ecocrop_limits_proposed.csv', index=False)

# ---- Part B: does water access explain present pixels outside the rainfall range? ----
rain = pix['chirps_annual_mean']
rows = []
for crop in ['rice', 'sugc', 'onio', 'toma', 'sorg', 'yams']:       # first four: suspected water-access crops; last two: rain-fed controls
    m = pix[f'present_{crop}'] > 0
    lo = req.loc[crop, 'rmin']
    below, inside = m & (rain < lo), m & (rain >= lo)
    rows.append({'crop': crop, 'n_present': int(m.sum()), 'n_below_abs_min': int(below.sum()),
                 'irrig_median_below': pix.loc[below, 'irrig_share'].median(),
                 'irrig_median_inside': pix.loc[inside, 'irrig_share'].median(),
                 'pct_irrig_gt20_below': (pix.loc[below, 'irrig_share'] > 0.2).mean() * 100,
                 'pct_irrig_gt20_inside': (pix.loc[inside, 'irrig_share'] > 0.2).mean() * 100,
                 'rho_share_vs_irrig': pix[f'share_{crop}'].corr(pix['irrig_share'], method='spearman')})
wa = pd.DataFrame(rows).set_index('crop')
print('\nWater access (irrig_share = irrigated share of harvested area in the SPAM pixel):')
print('Pixels with irrig_share > 20% over all pixels (%):', round(float((pix.irrig_share > 0.2).mean() * 100), 1))
print(wa.round(2).to_string())

# ---- Part C: independent field check of the rainfall limits (CropHarvest and surveys, not SPAM) ----
ch = pd.read_parquet(f'{P}/ground_truth_cropharvest_pixels.parquet')[['px', 'py', 'crop_code']].assign(source='CropHarvest')
sites = pd.read_parquet(f'{P}/ground_truth_surveys_sites.parquet')
scr = pd.read_parquet(f'{P}/ground_truth_surveys_crops.parquet')[['site_id', 'crop_code']]
use = sites[sites.within_5km & (sites.has_label == True)][['site_id', 'country', 'px', 'py']]
sv = scr.merge(use, on='site_id').rename(columns={'country': 'source'})[['px', 'py', 'crop_code', 'source']]
fo = pd.concat([ch, sv], ignore_index=True)
fo = fo[fo.crop_code.isin(CROPS)].copy()
fo[['px', 'py']] = fo[['px', 'py']].astype(int)
fo = fo.drop_duplicates(['px', 'py', 'crop_code', 'source'])
pixf = pix[['px', 'py', 'chirps_annual_mean']].astype({'px': int, 'py': int})
fo = fo.merge(pixf, on=['px', 'py'], how='inner')

rows = []
for (crop, src), d in fo.groupby(['crop_code', 'source']):
    if len(d) < 10:
        continue
    x = d.chirps_annual_mean.to_numpy('float64')
    a0, a1 = req.loc[crop, 'rmin'], req.loc[crop, 'rmax']
    w0, w1 = WIDE[(crop, 'rainfall')][0], WIDE[(crop, 'rainfall')][3]
    rows.append({'crop': crop, 'source': src, 'n_pixels': len(d),
                 'p05': np.percentile(x, 5), 'p50': np.percentile(x, 50), 'p95': np.percentile(x, 95),
                 'pct_below_ecocrop_min': (x < a0).mean() * 100, 'pct_above_ecocrop_max': (x > a1).mean() * 100,
                 'pct_outside_widened': ((x < w0) | (x > w1)).mean() * 100})
fc = pd.DataFrame(rows)
print('\nField-observed crop pixels (at least 10 per crop and source): rainfall against the limits')
print(fc.round(0).to_string(index=False))
print('\nField pixels per crop and source (all):')
print(fo.groupby(['crop_code', 'source']).size().unstack(fill_value=0).to_string())

drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 4: EcoCrop trapezoidal scores, water-access rules, and comparison with 09b
import json, warnings
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'
cfg = json.load(open(f'{P}/training_config.json'))
CROPS = cfg['crops']
SHARE = [f'share_{c}' for c in CROPS]
PRES = [f'present_{c}' for c in CROPS]
K, N_BOOT = 5, 300

# ---- Rules (from the checks in cells 2 and 3 and the water-access diagnostics) ----
WIDEN_RAIN_MAX = {'sorg': 1170.0}          # supported by field data (Part C)
RAIN_TERM_OFF = ['onio', 'toma', 'sugc']   # irrigation-dependent: rainfall term not used
WATER_ACCESS = ['rice']                    # rainfall term dropped when irrigation or floodplain access is available

req = pd.read_parquet(f'{P}/crop_requirements.parquet').set_index('crop').loc[CROPS]

# ---- Pixel table (same construction as notebooks 08, 09, 09b) ----
tab = pd.read_parquet(f'{P}/training_table.parquet',
                      columns=['px', 'py', 'fold', 'adm0', 'has_label', 'temp_mean',
                               'chirps_annual_mean', 'phh2o_top'] + SHARE + PRES)
lab = tab[tab.has_label]
g = lab.groupby(['px', 'py'])
pix = (g[['temp_mean', 'chirps_annual_mean', 'phh2o_top']].mean()
       .join(g[SHARE].first()).join(g[PRES].first()).join(g[['fold']].first())
       .join(g['adm0'].agg(lambda s: s.mode().iat[0])).reset_index())
Y = np.clip(pix[SHARE].to_numpy('float64'), 0, 1)
folds = pix.fold.to_numpy()
idx = {f'{int(a)}_{int(b)}': i for i, (a, b) in enumerate(zip(pix.px, pix.py))}
n = len(pix)

# ---- Out-of-fold predictions from notebooks 08 and 09 ----
rf = pix[['px', 'py']].merge(pd.read_parquet(f'{P}/oof_rf_env.parquet').drop(columns='fold'),
                             on=['px', 'py'], how='left')[CROPS].to_numpy('float64')
lg = pix[['px', 'py']].merge(pd.read_parquet(f'{P}/oof_lgb_env.parquet').drop(columns='fold'),
                             on=['px', 'py'], how='left')[CROPS].to_numpy('float64')

# ---- EcoCrop trapezoidal scores ----
def trap(x, a0, o0, o1, a1):
    """0 outside the absolute range, 1 inside the optimal range, linear in between."""
    return np.interp(x, [a0, o0, o1, a1], [0.0, 1.0, 1.0, 0.0], left=0.0, right=0.0)

def eco_terms(crop, water_access):
    r = req.loc[crop]
    t = pix.temp_mean.to_numpy('float64')
    terms = {'temp': np.where(np.isnan(t), np.nan, ((t >= r.tmin) & (t <= r.tmax)).astype(float))}  # absolute range only
    terms['ph'] = trap(pix.phh2o_top.to_numpy('float64'), r.phmin, r.phopmn, r.phopmx, r.phmax)
    rain_off = crop in RAIN_TERM_OFF or (water_access and crop in WATER_ACCESS)
    if not rain_off:
        a1 = WIDEN_RAIN_MAX.get(crop, r.rmax)
        terms['rain'] = trap(pix.chirps_annual_mean.to_numpy('float64'), r.rmin, r.ropmn, r.ropmx, a1)
    return terms

def eco_matrices(water_access):
    mn, me = np.zeros((n, len(CROPS))), np.zeros((n, len(CROPS)))
    for j, c in enumerate(CROPS):
        T = np.column_stack(list(eco_terms(c, water_access).values()))
        mn[:, j] = np.nan_to_num(np.nanmin(T, axis=1))   # limiting factor
        me[:, j] = np.nan_to_num(np.nanmean(T, axis=1))  # average suitability
    return mn, me

mn0, me0 = eco_matrices(water_access=False)   # rain-fed
mn1, me1 = eco_matrices(water_access=True)    # irrigation or floodplain available

models = {'RF (09)': rf,
          'LightGBM (09)': lg,
          'EcoCrop min, rain-fed': mn0,
          'EcoCrop min, water access': mn1,
          'EcoCrop mean, rain-fed': me0,
          'LightGBM x EcoCrop min, rain-fed': lg * mn0,
          'LightGBM x EcoCrop min, water access': lg * mn1}

# ---- 1. Agreement with SPAM (pixel level) ----
def spam_metrics(M):
    ob = np.zeros(Y.shape, bool); np.put_along_axis(ob, np.argsort(-Y, 1, kind='stable')[:, :K], True, 1)
    pb = np.zeros(Y.shape, bool); np.put_along_axis(pb, np.argsort(-M, 1, kind='stable')[:, :K], True, 1)
    rho = np.nanmean([spearmanr(Y[:, j], M[:, j]).correlation for j in range(len(CROPS))])
    return {'SPAM rank corr': rho,
            'SPAM top-5 overlap': (ob & pb).sum(1).mean() / K,
            'SPAM top-1 match': (np.argmax(M, 1) == np.argmax(Y, 1)).mean()}

spam_rows = {name: spam_metrics(M) for name, M in models.items()}

# ---- 2. Survey presence AUC with bootstrap (as in 09b) ----
sites = pd.read_parquet(f'{P}/ground_truth_surveys_sites.parquet')
scrops = pd.read_parquet(f'{P}/ground_truth_surveys_crops.parquet')
use = sites[sites.within_5km & (sites.has_label == True)].copy()
use['pixel'] = use.px.astype(int).astype(str) + '_' + use.py.astype(int).astype(str)
use = use[use.pixel.isin(list(idx))]
full = (use[['site_id', 'country', 'pixel']].assign(_k=1)
        .merge(pd.DataFrame({'crop_code': CROPS, '_k': 1}), on='_k').drop(columns='_k')
        .merge(scrops[['site_id', 'crop_code', 'weight_share']], on=['site_id', 'crop_code'], how='left')
        .fillna({'weight_share': 0.0}))
obs = full.groupby(['country', 'pixel', 'crop_code']).weight_share.mean().unstack('crop_code')[CROPS]

rng = np.random.default_rng(0)
sur = []
for country in ['Niger', 'Nigeria']:
    o = obs.xs(country, level='country')
    ii = np.array([idx[p] for p in o.index])
    for j, c in enumerate(CROPS):
        y = o[c].to_numpy() > 0
        if y.sum() < 10 or (~y).sum() < 10:
            continue
        for name, M in models.items():
            s = M[ii, j]
            if s.std() == 0:
                continue
            vals = []
            for _ in range(N_BOOT):
                b = rng.integers(0, len(y), len(y))
                if 0 < y[b].sum() < len(y) and s[b].std() > 0:
                    vals.append(roc_auc_score(y[b], s[b]))
            lo, hi = np.percentile(vals, [2.5, 97.5])
            sur.append({'model': name, 'country': country, 'crop': c,
                        'auc': roc_auc_score(y, s), 'ci_low': lo, 'ci_high': hi})
sur = pd.DataFrame(sur)
sur_sum = (sur.groupby('model')
           .agg(mean_auc=('auc', 'mean'), crops_ci_above_05=('ci_low', lambda s: int((s > 0.5).sum())),
                n_crop_country=('auc', 'size')))

# ---- 3. CropHarvest top-5 hit rate (as in 09b) ----
ch = pd.read_parquet(f'{P}/ground_truth_cropharvest_pixels.parquet')
ch['pixel'] = ch.px.astype(int).astype(str) + '_' + ch.py.astype(int).astype(str)
ch = ch[ch.pixel.isin(list(idx)) & ch.crop_code.isin(CROPS)].copy()
ci = ch.pixel.map(idx).to_numpy()
cj = ch.crop_code.map({c: j for j, c in enumerate(CROPS)}).to_numpy()
EASY = ['sorg', 'pmil', 'cowp', 'grou', 'maiz']
is_easy = ch.crop_code.isin(EASY).to_numpy()
ranks = {name: (-M).argsort(1).argsort(1) for name, M in models.items()}

def hit(name, mask, crop=None):
    r = ranks[name][ci[mask], cj[mask]]
    return (r < K).mean()

cha = pd.DataFrame({
    'top5_all': {k: hit(k, np.ones(len(ci), bool)) for k in models},
    'top5_excl_common': {k: hit(k, ~is_easy) for k in models},
    'top5_common': {k: hit(k, is_easy) for k in models},
    'top5_sorghum': {k: hit(k, ch.crop_code.to_numpy() == 'sorg') for k in models},
    'top5_rice': {k: hit(k, ch.crop_code.to_numpy() == 'rice') for k in models}})

# ---- 4. Rice in dry pixels: how often does each rule put rice in the top band? ----
dry = (pix.chirps_annual_mean.to_numpy() < 1000) & (pix.present_rice.to_numpy() > 0)
rice_j = CROPS.index('rice')
rice_dry = pd.Series({
    'rice-present pixels below 1000 mm': int(dry.sum()),
    'rain-fed: share with score >= 0.5': float((mn0[dry, rice_j] >= 0.5).mean()),
    'water access: share with score >= 0.5': float((mn1[dry, rice_j] >= 0.5).mean())})

# ---- Results ----
spam_df = pd.DataFrame(spam_rows).T.round(3)
pd.set_option('display.width', 240)
print('1. Agreement with SPAM (out-of-fold or EcoCrop, all labelled pixels)')
print(spam_df.to_string())
print('\n2. Survey presence AUC (Niger and Nigeria, crops with 10+ pixels per class)')
print(sur_sum.round(3).to_string())
print('\n3. CropHarvest top-5 hit rate (chance 0.31)')
print(cha.round(3).to_string())
print('\n4. Rice in rainfall-dry labelled pixels')
print(rice_dry.round(3).to_string())
print('\nSurvey AUC by crop (widest gap between EcoCrop and LightGBM, both countries):')
piv = sur.pivot_table(index=['country', 'crop'], columns='model', values='auc')
if 'LightGBM (09)' in piv and 'EcoCrop min, rain-fed' in piv:
    piv['gap'] = piv['EcoCrop min, rain-fed'] - piv['LightGBM (09)']
    print(piv.sort_values('gap').round(2).to_string())

out = pd.concat({'spam': spam_df,
                 'survey': sur_sum.set_index(sur_sum.index),
                 'cropharvest': cha.T.round(3)}, axis=0)
out.to_csv(f'{P}/results_ecocrop_cell4.csv')
sur.to_csv(f'{P}/results_ecocrop_survey_auc.csv', index=False)
pd.DataFrame(mn1, columns=CROPS).assign(grid_pixel=pix.px.astype(str) + '_' + pix.py.astype(str)) \
  .to_parquet(f'{P}/ecocrop_scores_pixel.parquet', index=False)

drive.flush_and_unmount(); drive.mount('/content/drive')

In [ ]:
# @title Cell 5: final EcoCrop flag rules, app file and README text
import json
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
P = f'{BASE}/crop_project'
cfg = json.load(open(f'{P}/training_config.json'))
CROPS = cfg['crops']
req = pd.read_parquet(f'{P}/crop_requirements.parquet').set_index('crop').loc[CROPS]

FLAG_THRESHOLD = 0.5   # limiting-factor score below this raises a flag

# Rules fixed from the Cell 3 and Cell 4 results
RULES = {
    'sorg': {'rain_max': 1170, 'note': 'Field data show sorghum above the original 700 mm maximum in Nigeria and Burkina Faso; the maximum is widened to 1,170 mm.'},
    'rice': {'rain_min_applies': 'unless irrigation or floodplain access is available',
             'note': 'Rice is grown without irrigation in dry areas (Niger, Burkina Faso). Rainfall below 1,000 mm needs irrigation or floodplain water; the app asks for this.'},
    'onio': {'rain_term': False, 'note': 'Dry-season irrigated crop; rainfall is not checked.'},
    'toma': {'rain_term': False, 'note': 'Dry-season irrigated crop; rainfall is not checked.'},
    'sugc': {'rain_term': False, 'note': 'Irrigation-dependent; rainfall is not checked.'},
    'yams': {'note': 'Rainfall minimum of 800 mm rests on SPAM only; no field pixels to validate it.'},
}
TERMS = ['temperature', 'rainfall', 'ph']

flags = {}
for crop in CROPS:
    r = req.loc[crop]
    rule = RULES.get(crop, {})
    terms = {'temperature': {'abs_min': r.tmin, 'abs_max': r.tmax},
             'ph': {'abs_min': r.phmin, 'abs_max': r.phmax}}
    if rule.get('rain_term', True):
        terms['rainfall'] = {'abs_min': r.rmin,
                             'abs_max': rule.get('rain_max', r.rmax)}
    flags[crop] = {
        'name': str(r.ecocrop_name),
        'terms': terms,
        'rainfall_check': 'rain_term' if rule.get('rain_term', True) else 'none',
        'rice_water_access': rule.get('rain_min_applies') if crop == 'rice' else None,
        'flag_threshold': FLAG_THRESHOLD,
        'note': rule.get('note', ''),
        'source': 'FAO EcoCrop (via OpenCLIM/ecocrop v1.0); ranges widened only where stated'}

with open(f'{P}/ecocrop_flags.json', 'w') as f:
    json.dump({'threshold': FLAG_THRESHOLD, 'crops': flags}, f, indent=1)

# Summary of how often each flag fires over labelled pixels (rain-fed unless the rice rule applies)
ev = pd.read_parquet(f'{P}/training_table.parquet', columns=['has_label', 'temp_mean', 'chirps_annual_mean', 'phh2o_top'])
ev = ev[ev.has_label].groupby(level=0).size() if False else ev[ev.has_label]
rows = []
for crop in CROPS:
    f = flags[crop]
    fired = pd.Series(False, index=ev.index)
    for var, col in [('temperature', 'temp_mean'), ('ph', 'phh2o_top'), ('rainfall', 'chirps_annual_mean')]:
        if var in f['terms']:
            lo, hi = f['terms'][var]['abs_min'], f['terms'][var]['abs_max']
            x = ev[col]
            fired |= (x < lo) | (x > hi)
    rows.append({'crop': crop, 'name': f['name'], 'pct_pixels_flagged': round(float(fired.mean() * 100), 1),
                 'rain_check': f['rainfall_check']})
summary = pd.DataFrame(rows).set_index('crop')
print(summary.to_string())

drive.flush_and_unmount(); drive.mount('/content/drive')